In [ ]:
import pandas as pd
import re
import os
from pathlib import Path

In [ ]:
histogram_path = "Patient1/Xenium_1_Patient_1_Histogram_Values.csv"
measurements_path = "Patient1/Xenium_1_Patient_1_Measurements.csv"
#
hist_df = pd.read_csv(histogram_path)
meas_df = pd.read_csv(measurements_path)

In [5]:
for col in meas_df.columns:
    print(col)


Image
Object ID
Object type
Name
Classification
Parent
ROI
Centroid X µm
Centroid Y µm
Nucleus: Area
Nucleus: Perimeter
Nucleus: Circularity
Nucleus: Max caliper
Nucleus: Min caliper
Nucleus: Eccentricity
Nucleus: DNA1 mean
Nucleus: DNA1 sum
Nucleus: DNA1 std dev
Nucleus: DNA1 max
Nucleus: DNA1 min
Nucleus: DNA1 range
Nucleus: CHECK mean
Nucleus: CHECK sum
Nucleus: CHECK std dev
Nucleus: CHECK max
Nucleus: CHECK min
Nucleus: CHECK range
Nucleus: HSV1-rp mean
Nucleus: HSV1-rp sum
Nucleus: HSV1-rp std dev
Nucleus: HSV1-rp max
Nucleus: HSV1-rp min
Nucleus: HSV1-rp range
Nucleus: CD8 mean
Nucleus: CD8 sum
Nucleus: CD8 std dev
Nucleus: CD8 max
Nucleus: CD8 min
Nucleus: CD8 range
Nucleus: DNA2 mean
Nucleus: DNA2 sum
Nucleus: DNA2 std dev
Nucleus: DNA2 max
Nucleus: DNA2 min
Nucleus: DNA2 range
Nucleus: HSV1 mean
Nucleus: HSV1 sum
Nucleus: HSV1 std dev
Nucleus: HSV1 max
Nucleus: HSV1 min
Nucleus: HSV1 range
Nucleus: OLIG2 mean
Nucleus: OLIG2 sum
Nucleus: OLIG2 std dev
Nucleus: OLIG2 max
Nucleu

In [ ]:
root_dir = Path("J:/long/CyCIF/")

# Loop through patient folders
for patient_folder in root_dir.glob("Patient*"):
    # Detect files
    hist_file, meas_file = None, None
    for file in patient_folder.iterdir():
        if "Histogram_Values.csv" in file.name:
            hist_file = file
        elif "Measurements.csv" in file.name:
            meas_file = file
    
    # Skip if either file missing
    if hist_file is None or meas_file is None:
        print(f"Skipping {patient_folder}, missing required files.")
        continue
    
    # Output name
    out_prefix = meas_file.name.replace("_Measurements.csv", "")
    output_path = patient_folder / f"{out_prefix}_Phenotyped.csv"
    
    # Load files
    hist_df = pd.read_csv(hist_file)
    meas_df = pd.read_csv(meas_file)
    
    # Normalize measurement column lookup (remove spaces + lowercase)
    normalized_cols = {re.sub(r"\s+", "", c).lower(): c for c in meas_df.columns}
    
    # Metadata columns to keep
    cols_to_keep = [
        "Object ID", "Object type", "Image",
        "Centroid X µm", "Centroid Y µm",
        "Nucleus/Cell area ratio", "Parent", "ROI",
        "Nucleus: Area", "Nucleus: Perimeter", "Nucleus: Circularity",
        "Nucleus: Max caliper", "Nucleus: Min caliper", "Nucleus: Eccentricity",
        "Cell: Area", "Cell: Perimeter", "Cell: Circularity",
        "Cell: Max caliper", "Cell: Min caliper", "Cell: Eccentricity"
    ]
    phenotype_df = meas_df[cols_to_keep].copy()
    
    # Collect phenotype columns here
    binarized_cols = []
    
    # Apply thresholds for binary positivity
    for _, row in hist_df.iterrows():
        marker = row["Marker"]
        clean_marker = re.sub(r"\s+", "", marker).lower()   # normalize name
        min_val, max_val = row["Channel Min"], row["Channel Max"]
        
        # Special case: HSV1 markers -> include both HSV1 and HSV1-rp
        if "hsv1" in clean_marker:
            candidates = [c for c in normalized_cols.keys() if "hsv1" in c]
        else:
            candidates = [c for c in normalized_cols.keys() if clean_marker in c]
        
        if not candidates:
            print(f"{patient_folder.name}: could not match marker {marker}")
            continue
        
        # Restrict to mean only
        candidates = [c for c in candidates if "mean" in c]
        
        # Apply histogram threshold to all compartments
        for candidate in candidates:
            col_name = normalized_cols[candidate]
            
            # Parse compartment and marker from original column name
            # e.g. "Cytoplasm: HSV1 mean" -> Cytoplasm_HSV1_mean_bin
            parts = col_name.split(":")
            if len(parts) == 2:
                compartment, rest = parts[0].strip(), parts[1].strip()
                rest = rest.replace(" ", "_")  # "HSV1 mean" → "HSV1_mean"
                new_col = f"{compartment}_{rest}_bin"
            else:
                new_col = f"{col_name.replace(':','_').replace(' ','_')}_bin"
            
            binarized = (
                (meas_df[col_name] >= min_val) & (meas_df[col_name] <= max_val)
            ).astype(int)
            
            binarized_cols.append(pd.DataFrame({new_col: binarized}))
    
    # Concatenate all binarized marker columns 
    if binarized_cols:
        phenotype_df = pd.concat([phenotype_df] + binarized_cols, axis=1)
    
    # Save output
    phenotype_df.to_csv(output_path, index=False)
    print(f"✅ Saved phenotyped file: {output_path} "
          f"({phenotype_df.shape[0]} cells × {phenotype_df.shape[1]} cols)")

✅ Saved phenotyped file: J:\long\CyCIF\Patient1\Xenium_1_Patient_1_Phenotyped.csv (52293 cells × 107 cols)
✅ Saved phenotyped file: J:\long\CyCIF\Patient2\Xenium_2_Patient_2_Phenotyped.csv (52752 cells × 80 cols)
✅ Saved phenotyped file: J:\long\CyCIF\Patient3\Xenium_2_Patient_3_Phenotyped.csv (94519 cells × 77 cols)
✅ Saved phenotyped file: J:\long\CyCIF\Patient4\Xenium_2_Patient_4_Phenotyped.csv (64549 cells × 80 cols)
✅ Saved phenotyped file: J:\long\CyCIF\Patient5\Xenium_1_Patient_5_Phenotyped.csv (21355 cells × 98 cols)
✅ Saved phenotyped file: J:\long\CyCIF\Patient6\Xenium_1_Patient_6_Phenotyped.csv (19321 cells × 101 cols)


In [ ]:
##
#  Patient 6 histogram csv file is saved with tab delimiter, need to convert to comma separated
##
file_path = "Patient6/Xenium_1_Patient_6_Histogram_Values.csv"
output_path = "Patient6/Xenium_1_Patient_6_Histogram_Values.csv"

# Read
hist6_df = pd.read_csv(file_path, sep="\t")

# Save as proper comma-separated CSV
hist6_df.to_csv(output_path, index=False)
